# Continual refine on the durable corpus

**What this shows:** how human-verified camera↔LiDAR pairs accumulate into XCalib's
**durable corrections corpus**, and how `run_continual_refine` trains the matcher from
that corpus and publishes a new revision only when it beats the current model on held-out
data. Everything runs inside the `xcalib` package; no annotation tool is needed.

XCalib has two ways to learn from a human:

| Tier | Call | State | When |
| --- | --- | --- | --- |
| **One-shot** | `session.bank_pairs(...)` → `session.adapt()` | in-memory, ephemeral, no held-out check | "nudge the model right now" |
| **Continual** | `record_verified_pairs(...)` → corpus → `run_continual_refine(...)` | durable sqlite, gated, accumulates | "keep improving over sessions" |

This notebook walks the **continual** tier end-to-end on the committed A9 sample:

1. build a SUSTechPOINTS-layout `Scene` from the A9 frames, with a base model for it,
2. `match_review` a frame (the matcher proposes one-to-one pairs),
3. `record_verified_pairs` → the corpus,
4. inspect the corpus,
5. `scene_frame_loader` — the frame reader refine uses,
6. `run_continual_refine` — gated training over the corpus,
7. load the published revision.

Run it from a checkout of the repository: it reads `demo/frames/a9_sample`. The released A9
weights come from the model hub on first use, or from
`checkpoints/crlite_a9_dataset_r02_s01_best.pth` if you keep a copy there. Refine runs on the device the saved
model config names, which is the CPU here and is enough for this sample.

You can do the review steps **either by hand in the `xcalib-review` window** (launched from a
cell below; needs `pip install "xcalib[gui]"` and a display) **or in code**. Both write to the
same corpus.

In [ ]:
import json
import os
import shutil
import sys
import tempfile
import uuid
from pathlib import Path

import numpy as np

from xcalib import Matcher
from xcalib.continual import (
    VerifiedPair, record_verified_pairs, scene_frame_loader,
    open_store, run_continual_refine, RefineConfig, model_dir, resolve_model, FeatureWork,
)
from xcalib.gui.scene import Scene, match_review

# On Windows, activating a conda or pixi environment puts its DLL folder on PATH and sets the
# OpenMP variable. A kernel started without activation (an editor's interpreter picker) has
# neither: the first NumPy matrix product then ends the kernel with exit code 0xC06D007F, and
# two OpenMP runtimes can abort it with "OMP: Error #15". These lines only fill in what
# activation sets; the OpenMP variable is an unsupported workaround, so it is only a default.
if sys.platform == "win32":
    _dlls = os.path.join(sys.prefix, "Library", "bin")
    if os.path.isdir(_dlls) and _dlls.lower() not in os.environ.get("PATH", "").lower():
        os.environ["PATH"] = _dlls + os.pathsep + os.environ.get("PATH", "")
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

DEMO = Path.cwd() if Path.cwd().name == "demo" else Path.cwd() / "demo"
A9 = DEMO / "frames" / "a9_sample"
# Optional local copy of the released A9 weights; without it the weights come from the model hub.
LOCAL_CKPT = DEMO.parent / "checkpoints" / "crlite_a9_dataset_r02_s01_best.pth"
CAMERA = "cam"
FRAMES = sorted(p.name for p in A9.iterdir() if p.is_dir())[:8]
print(f"{len(FRAMES)} A9 sample frames: {FRAMES}")

## 1. Build a `Scene` and a base model

The corpus, the loader, and the review window all read the **SUSTechPOINTS on-disk
layout** (`camera/`, `lidar/`, `label/`, `label_fusion/`). The committed A9 sample is
in the demo's own `detections.json` format, so we convert a few frames: 3D detections
(axis-aligned extents) become `psr` boxes with a stable LiDAR id, and 2D detections
become un-id'd camera rects the matcher will pair to a box.

Refine needs a base model in the scene's **model directory**
(`<data_root>/<scene>/calib/xcalib/<camera>_model/`). We save the released matcher there
with `save_pretrained`, which writes `crlite.pth` and `crlite.yaml`. Refine and the review
window both load the base model from this one location, and the corpus (`refine.sqlite`)
is created beside it.

In [ ]:
def build_scene(root: Path, frames: list[str]) -> Scene:
    for sub in ("camera/" + CAMERA, "lidar", "label", "label_fusion/camera/" + CAMERA):
        (root / sub).mkdir(parents=True, exist_ok=True)
    for frame in frames:
        det = json.loads((A9 / frame / "detections.json").read_text())
        shutil.copyfile(A9 / frame / "image.png", root / "camera" / CAMERA / f"{frame}.png")
        shutil.copyfile(A9 / frame / "point_cloud.pcd", root / "lidar" / f"{frame}.pcd")
        boxes = []
        for j, e in enumerate(det["bboxes_3d"]):
            lo, hi = np.asarray(e[:3], float), np.asarray(e[3:], float)
            c, s = (lo + hi) / 2.0, (hi - lo)
            boxes.append({"obj_id": str(j + 1), "obj_type": "Car",
                          "psr": {"position": {"x": c[0], "y": c[1], "z": c[2]},
                                  "scale": {"x": s[0], "y": s[1], "z": s[2]},
                                  "rotation": {"x": 0.0, "y": 0.0, "z": 0.0}}})
        (root / "label" / f"{frame}.json").write_text(json.dumps({"objs": boxes}))
        rects = [{"rect": {"x1": r[0], "y1": r[1], "x2": r[2], "y2": r[3]},
                  "obj_attr": f"t{i}", "obj_type": "Car"}
                 for i, r in enumerate(det["bboxes_2d"])]
        (root / "label_fusion" / "camera" / CAMERA / f"{frame}.json").write_text(
            json.dumps({"objs": rects}))
    return Scene(str(root), CAMERA)


WORK = Path(tempfile.mkdtemp(prefix="xcalib_refine_demo_"))
DATA_ROOT = WORK / "data"
scene = build_scene(DATA_ROOT / "site", FRAMES)

# Base model: a local checkpoint when present, otherwise the model hub.
matcher = Matcher.from_pretrained("crlite", weights=LOCAL_CKPT if LOCAL_CKPT.exists() else None,
                                  site="a9_dataset_r02_s01", device="cpu")
MDIR = model_dir(DATA_ROOT, "site", CAMERA)
matcher.save_pretrained(MDIR)

print("scene at   ", scene.root)
print("model dir  ", MDIR)
print("frames     ", scene.frames()[:8])

## Drive it in the review GUI (optional, local only)

Running locally **with a display** and `pip install "xcalib[gui]"`? Launch the real
`xcalib-review` window right here. IPython's `%gui qt` runs the Qt event loop in the
background, so the window stays live while the kernel is free.

In the window: step frames (‹ ›), fix any wrong pair (click a
card → pick the right #id or *— none —*), **Apply this frame**, then **Refine
(accumulate)** — which records the applied decisions to the corpus and runs the gated refine
with the default gate. On a small scene like this one the default gate usually holds the run
back (outcome `rejected`, reason `insufficient grouped validation`); the code cells below
run the same refine with a gate relaxed for the demo. Close the window when done, then jump
to *4. Inspect the corpus* to see what it wrote.

Do the review **either** by hand here **or** in code. On a headless machine, skip this cell.

In [ ]:
# LOCAL ONLY — opens a live Qt window; skip on headless machines.
%gui qt
from PySide6.QtWidgets import QApplication
from xcalib.gui.review import ReviewWindow

_app = QApplication.instance() or QApplication([])
_win = ReviewWindow(scene, device="cpu")  # loads the base model in a worker thread
_win.show()
_win.raise_()
print("review window opened — review, Apply, then click 'Refine (accumulate)'.")

## …or step through it in code

The cells below do the same steps as the window, headless — handy for CI or just to see the
API. A decision you saved in the window (Apply) wins over the matcher's pick, so running
both, in either order, keeps your corrections.

## 2. Match a frame

`match_review` runs the matcher and returns one row per camera
rect: its ranked candidate boxes and the one-to-one **suggestion** (the id the
matcher would assign). This is what the review window shows as cards. A box whose crop
holds no LiDAR points cannot be scored: it stays in every menu with score 0.0 so you can
still pick it, and it is never suggested. The geometric gate applies only when the scene has a calibration
(`calib/camera/<camera>.json`); this demo scene has none.

In [ ]:
review = match_review(matcher, scene, FRAMES[0], threshold=0.3)
for t in review.tracks[:6]:
    top = t.options[0] if t.options else None
    if top is None:
        print(f"rect {t.key}: no options")
    else:
        print(f"rect {t.key}: suggestion={t.suggestion or '—'} "
              f"top=({top['lidar_id']}@{top['score']:.2f})")

## 3. Record the verified pairs into the corpus

A human would accept or correct each suggestion. Here a decision already saved in the window
(an applied id, or an explicit *— none —*) is recorded as it is, and every other rect takes
the matcher's pick when it has one. `record_verified_pairs` upserts the pairs into the
durable corpus, keyed by a canonical observation key (scene, camera, frame, rect) and stamped
with a fingerprint of the frame as it is on disk now.

`proposal` is the matcher's own pick. It sets `provenance` (`verified_model_correct` when the
recorded pair equals the proposal, otherwise `human_corrected`), and it tells training which
box a human replaced.

We open the store at the model dir the refine step will resolve
(`<data_root>/<scene>/calib/xcalib/<camera>_model/refine.sqlite`).

In [ ]:
def pairs_from_review(review) -> list[VerifiedPair]:
    out = []
    for t in review.tracks:
        track = t.key if t.key.isdigit() else None  # match_review keys tracks by digits
        candidates = [{"lidar_key": "id:" + o["lidar_id"], "score": o["score"]}
                      for o in t.options]
        proposal = ({"decision": "positive", "lidar_key": "id:" + t.suggestion}
                    if t.suggestion else None)
        if t.applied_id:        # saved with Apply in the window
            decision, key = "positive", "id:" + t.applied_id
        elif t.rejected:        # saved as "— none —"
            decision, key = "no_match", None
        elif t.suggestion:      # nothing saved yet: accept the matcher's pick
            decision, key = "positive", "id:" + t.suggestion
        else:
            continue
        out.append(VerifiedPair(rect=t.rect, decision=decision, lidar_key=key, track=track,
                                candidates=candidates, proposal=proposal))
    return out

store = open_store(MDIR)  # the model dir set up above (base model + corpus live here)

total = 0
for frame in FRAMES:
    review = match_review(matcher, scene, frame, threshold=0.3)
    pairs = pairs_from_review(review)
    if pairs:
        total += record_verified_pairs(store, str(DATA_ROOT), "site", CAMERA, frame, pairs)
print(f"recorded {total} verified pairs into the corpus")
store.close()  # the next cell opens the corpus again, so it also works after reviewing in the window

## 4. Inspect the corpus

`iter_corrections` returns one page of the active corrections: at most 1000 rows per call
(pass `after_sample_key=` to continue). Each row is one reviewed observation — its stable
keys, the accepted target, and a `provenance` label recorded for audit.

In [ ]:
store = open_store(MDIR)  # also holds whatever the window recorded
active = list(store.iter_corrections())  # first page
print(f"active corrections (first page): {len(active)}")
row = active[0]
print("sample_key      :", row.sample_key)
print("observation_key :", row.observation_key)
print("frame / track   :", row.frame, "/", row.camera_track)
print("final_lidar_key :", row.final_lidar_key)
print("provenance      :", row.provenance)
print("fingerprint     :", row.input_fingerprint[:24], "…")

## 5. The frame loader

`run_continual_refine` reads each reviewed frame back through `scene_frame_loader` (the
default, built on `xcalib.gui.scene`). It re-derives the same stable `rect_keys` and
`lidar_keys` (a rect with no recorded pair gets a positional key `rect:<i>`) and checks that
the frame is unchanged. The fingerprint is recomputed from the image and point-cloud file
timestamps and sizes plus the box geometry; if it no longer matches the one recorded with the
pairs, the frame is skipped and marked stale. Editing boxes after recording, or copying the
scene without keeping file timestamps, does this. Record the frame again to refresh it.

In [ ]:
frame0 = active[0].frame
work = FeatureWork(frame=frame0, input_fingerprint=active[0].input_fingerprint,
                   dataset_split="train")
recs = [r for r in active if r.frame == frame0]
fi = scene_frame_loader(str(DATA_ROOT), "site", CAMERA, work, recs)
print("image        :", fi.image.shape, "| points:", fi.point_cloud.shape)
print("rect_keys[:4]:", fi.rect_keys[:4])
print("lidar_keys[:4]:", fi.lidar_keys[:4])
print("fingerprint matches the reviewed snapshot:",
      fi.input_fingerprint == active[0].input_fingerprint)

## 6. Continual refine (gated)

`run_continual_refine` resolves the base model from the model dir and works through these
phases: **indexing** the corpus, **caching** frozen-base embeddings for every reviewed frame
(through `scene_frame_loader`), **classifying** each sample as easy or hard for the current
model, **training** a warm-started candidate on the hard ones, **validating** the candidate
against the current model on held-out samples, **auditing** it on a second held-out split, and
**publishing** a new revision only if the gate passes. Otherwise nothing is published and the
current model keeps serving.

Samples are grouped by camera track and split by a stable hash into train (80%), validation
(10%) and audit (10%). The default gate needs at least 20 validation samples from at least 2
tracks and a primary-score gain of at least 0.005 over the current model. The 8 demo frames
fall well short of that, so the default gate rejects this run with
`insufficient grouped validation`. The cell below relaxes the gate (`min_validation_samples`,
`min_validation_groups`, `minimum_primary_delta`) so the run can publish on a handful of
frames; **never relax it for real data**. A `run_id` can publish only once, so the cell
generates a new one each time.

In [ ]:
store.close()  # run_continual_refine opens its own connection to refine.sqlite

demo_gate = RefineConfig(min_validation_samples=1, min_validation_groups=1,
                         minimum_primary_delta=-1.0)  # demo-relaxed; NOT for production

def show(p):
    if p.current in (0, p.total):
        print(f"  [{p.phase}] {p.message}")

run_id = f"demo-{uuid.uuid4().hex[:8]}"  # a run_id can publish only once
result = run_continual_refine(str(DATA_ROOT), "site", CAMERA, run_id,
                              show, config=demo_gate)
print("\noutcome        :", result.outcome)
print("new_revision   :", result.new_revision)
print("best_epoch     :", result.best_epoch)
print("gate_reasons   :", result.gate_reasons or "(none)")
print("remaining      :", result.remaining)
if result.validation_parent and result.validation_candidate:
    p, c = result.validation_parent, result.validation_candidate
    print(f"primary_score  : parent {p.primary_score:.3f} -> candidate {c.primary_score:.3f}")
    print(f"positive_top1  : parent {p.positive_top1:.3f} -> candidate {c.positive_top1:.3f}")

## 7. Use the published revision

An accepted run writes the revision to `<model dir>/revisions/<revision>/` (`crlite.pth` and
`crlite.yaml`) and points `<model dir>/current.json` at it. The base `crlite.pth` is left
untouched, and the next refine run starts from the serving revision. `resolve_model` returns
the serving revision, which you load with `Matcher.from_pretrained`.

`Matcher` loads the revision's adapters. The acceptance threshold that refine learned is
stored in the revision (`continual_state_dict`), but `Matcher` does not read it. The
`xcalib-review` window does not follow `current.json`; it keeps loading the `.pth` in the
model directory.

In [ ]:
ref = resolve_model(DATA_ROOT, "site", CAMERA)
print("serving revision:", ref.revision.revision, "| source:", ref.revision.source)
print("parent revision :", ref.revision.parent_revision)
print("weights         :", ref.revision.weights_path)
refined = Matcher.from_pretrained(ref.revision.model_name, weights=ref.revision.weights_path,
                                  config=ref.revision.config_path, device="cpu")

## Where the corpus lives, and what's next

- **Corpus location is an explicit parameter.** The store is opened at a *model dir*
  (`<data_root>/<scene>/calib/xcalib/<camera>_model/refine.sqlite`). Point it wherever you
  keep scenes.
- **One-shot is still there.** `matcher.oneshot(intr).bank_pairs(...)` → `adapt()`
  remains the fast, in-session tier with no held-out check. Continual refine is the durable,
  accumulating, gated one — this notebook.
- **In the review window:** `xcalib-review --scene <scene dir> --camera <camera>` exposes the
  same flow — step frames, fix pairs, Apply (writes the decisions to the scene labels), then
  **Refine (accumulate)** (records them to the corpus and refines).
- **Clean up:** the demo scene lives in a temporary directory. Remove it when you are done
  with `shutil.rmtree(WORK, ignore_errors=True)`.

The [continual refine guide](../docs/continual-refine.md) covers the corpus, the gate,
published revisions, and cancelling a run.